In [13]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="white", context="notebook")
pd.set_option('display.max_columns', None)

# Introduction

La régularité TGV tient compte des différentes durées de trajet des clients (aussi appelée composite).

- Un train est considéré à l'heure si son retard au terminus est inférieur à 5min pour un parcours inférieur à 1h30
- Un train est considéré à l'heure si son retard au terminus est inférieur à 10min pour un parcours entre 1h30 et 3h
- Un train est considéré à l'heure si son retard au terminus est inférieur à 15min pour un parcours au-delà de 3h

Les horaires d'arrivée sont également déterminés par des capteurs détectant le passage du train à un point déterminé marquant l'entrée en gare et exceptionnellement par des suivis manuel. La précision des mesures est la minute arrondie à minute inférieure, ce qui est conforme à l'ensemble des normes retenues pour la confection des horaires et chronogrammes de service.


Le résultat de régularité global n’est ni la moyenne des résultats des 6 axes, ni la moyenne des résultats de l’ensemble des liaisons. En effet, le taux de régularité est le nombre de trains à l’heure à leur terminus sur le nombre de trains total ayant circulé sur le périmètre considéré. De plus, un même TGV peut compter dans plusieurs liaisons, mais il ne compte qu’une seule fois dans la régularité globale. Les chiffres de ce jeu de données ne doivent donc pas être utilisés pour calculer une régularité globale TGV ou par axe.

In [14]:
df = pd.read_csv("regularite-mensuelle-tgv-aqst.csv", sep=";")
df.head(5)

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Commentaire retards au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,Retard moyen de tous les trains à l'arrivée,Commentaire retards à l'arrivée,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,3.693179,NaN,147,28.436735,6.511118,NaN,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,4.567119,NaN,105,18.049048,5.363539,"Ce mois-ci, l'OD a été touchée par les inciden...",32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,0.286283,NaN,19,24.736842,2.938053,NaN,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,0.734290,NaN,58,33.726437,5.292211,NaN,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,3.229701,NaN,89,14.592697,4.882372,NaN,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


Une ligne de notre tableau de donnée correspond à un voyage de notre échantillon.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12544 entries, 0 to 12543
Data columns (total 26 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   Date                                                                                         12544 non-null  object 
 1   Service                                                                                      12544 non-null  object 
 2   Gare de départ                                                                               12544 non-null  object 
 3   Gare d'arrivée                                                                               12544 non-null  object 
 4   Durée moyenne du trajet                                                                      12544 non-null  int64  
 5   Nombre de circulations prévues  

On compte 12544 lignes et 26 colonnes dans notre tableau de données.

In [11]:
df.isna().sum()

Date                                                                                               0
Service                                                                                            0
Gare de départ                                                                                     0
Gare d'arrivée                                                                                     0
Durée moyenne du trajet                                                                            0
Nombre de circulations prévues                                                                     0
Nombre de trains annulés                                                                           0
Commentaire annulations                                                                        12544
Nombre de trains en retard au départ                                                               0
Retard moyen des trains en retard au départ                                                

On constate grâce aux infos globales de nos données que la colonne :

- "Commentaire annulations" ne contient que des valeurs vides (NaN) sur l'ensemble des $12\,544$ lignes.
- "Commentaire retards au départ" ne contient que des valeurs vides (NaN) sur l'ensemble des $12\,544$ lignes.

Aussi ces deux colonnes sont donc totalement inutiles pour notre analyse, car elles ne contiennent aucune donnée exploitable.

En revanche, la colonne "Commentaire retards à l'arrivée" contient des informations pour certaines lignes (698 valeurs renseignées). Elle peut être intéressante si on souhaite analyser qualitativement les causes de perturbations.